# CWRU 10-Class MLR
- Window sizes: `256, 512, 1024, 2048`
- Raw sample precisions: `4, 8, 12, 16` bits

- `logistic_params.h` — all 16 trained HLS parameter/coefficient sets behind compile-time `MODEL_W<window>_B<bits>` selectors.
- `experiment_data.npz` — all test windows, labels, features, model parameters, fixed-point parameters, and packing metadata for PYNQ.

In [1]:
%pip install matplotlib scikit-learn scipy pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import shutil
import urllib.request
import zipfile
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    StratifiedGroupKFold,
)
from scipy.signal import resample_poly


# ============================================================
# Global experiment settings
# ============================================================

RANDOM_STATE=42

REPO_DIR=Path('CWRU_Bearing_NumPy')
DATA_DIR=REPO_DIR/'Data'

PROTOCOL='all_loads'
RPM_LIST=[1797,1772,1750,1730] if PROTOCOL=='all_loads' else [1797]

# The notebook sweeps ALL combinations automatically.
SUPPORTED_WINDOW_SIZES=(256,512,1024,2048)
SUPPORTED_RAW_BITS=(4,8,12,16)

NUM_INPUT_BUSES=4
BUS_BITS=64

STRIDE=16
TRAIN_FRACTION=0.70
WINDOWS_PER_CLASS_PER_RPM=500
TRAIN_WINDOWS_PER_RECORD=int(
    WINDOWS_PER_CLASS_PER_RPM*TRAIN_FRACTION
)
TEST_WINDOWS_PER_RECORD=(
    WINDOWS_PER_CLASS_PER_RPM-
    TRAIN_WINDOWS_PER_RECORD
)

DOWNSAMPLE_NORMAL_TO_12K=True

#   4-bit  -> Q4.0
#   8-bit  -> Q4.4
#   12-bit -> Q4.8
#   16-bit -> Q4.12
SAMPLE_INT_BITS=4

# Feature integer width is fixed at 12 so the largest zero-crossing count
# (2047 for a 2048-sample window) is representable. The feature fractional
# width follows the raw sample fractional width for each precision.
FEATURE_INT_BITS=12

WEIGHT_TOTAL_BITS=18
BIAS_TOTAL_BITS=18

FEATURE_NAMES=['Peak','MAV','ZeroCrossing']

CLASS_SPECS=[
    (0,'Normal',None,None),
    (1,'B007','B',7),
    (2,'IR007','IR',7),
    (3,'OR007','OR@6',7),
    (4,'B014','B',14),
    (5,'IR014','IR',14),
    (6,'OR014','OR@6',14),
    (7,'B021','B',21),
    (8,'IR021','IR',21),
    (9,'OR021','OR@6',21),
]

CLASS_NAMES=[x[1] for x in CLASS_SPECS]
NUM_CLASSES=len(CLASS_NAMES)
NUM_FEATURES=len(FEATURE_NAMES)

# Same hyperparameter grid used by the previous workflow.
C_VALUES=[
    50.0,
    100.0,
    150.0,
    200.0,
    250.0,
    300.0,
    400.0,
    500.0,
    750.0,
    1000.0,
]

print('Protocol:',PROTOCOL)
print('RPMs:',RPM_LIST)
print('Window sweep:',SUPPORTED_WINDOW_SIZES)
print('Precision sweep:',SUPPORTED_RAW_BITS)
print(
    'Total configurations:',
    len(SUPPORTED_WINDOW_SIZES)*len(SUPPORTED_RAW_BITS)
)


Protocol: all_loads
RPMs: [1797, 1772, 1750, 1730]
Window sweep: (256, 512, 1024, 2048)
Precision sweep: (4, 8, 12, 16)
Total configurations: 16


### Fixed-point precision sweep

## Download / locate the NumPy CWRU repository

The dataset is downloaded only if it is not already present.


In [3]:
if not DATA_DIR.exists():
    zp=Path('CWRU_Bearing_NumPy-main.zip')
    url=(
        'https://github.com/srigas/'
        'CWRU_Bearing_NumPy/archive/refs/heads/main.zip'
    )

    urllib.request.urlretrieve(url,zp)

    with zipfile.ZipFile(zp,'r') as zf:
        zf.extractall('.')

    extracted=Path('CWRU_Bearing_NumPy-main')

    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    extracted.rename(REPO_DIR)
    zp.unlink(missing_ok=True)

assert DATA_DIR.exists()
print(DATA_DIR.resolve())


C:\ECEN296\cwru_precision_sweep_64bit_single_header\CWRU_Bearing_NumPy\Data


## Load the source recordings once


In [4]:
def source_path(rpm,fault,diam):
    folder=DATA_DIR/f'{rpm} RPM'

    if fault is None:
        return folder/f'{rpm}_Normal.npz'

    return folder/f'{rpm}_{fault}_{diam}_DE12.npz'


def flatten_series(v):
    a=np.asarray(v)

    if a.dtype==object:
        return np.concatenate([
            np.asarray(
                x,
                dtype=np.float64
            ).reshape(-1)
            for x in a.reshape(-1)
        ])

    return np.asarray(
        a,
        dtype=np.float64
    ).reshape(-1)


def load_de(path):
    with np.load(
        path,
        allow_pickle=True
    ) as d:
        if 'DE' not in d.files:
            raise KeyError(
                f'{path.name}: no DE; keys={d.files}'
            )

        return flatten_series(d['DE'])


MAX_REQUIRED_WINDOW=max(SUPPORTED_WINDOW_SIZES)

records=[]

for rpm in RPM_LIST:
    for label,name,fault,diam in CLASS_SPECS:
        p=source_path(
            rpm,
            fault,
            diam
        )

        if not p.exists():
            raise FileNotFoundError(p)

        sig=load_de(p)

        if (
            name=='Normal' and
            DOWNSAMPLE_NORMAL_TO_12K
        ):
            sig=resample_poly(
                sig,
                up=1,
                down=4
            )

        split=int(
            len(sig)*TRAIN_FRACTION
        )

        if (
            split<MAX_REQUIRED_WINDOW or
            len(sig)-split<MAX_REQUIRED_WINDOW
        ):
            raise ValueError(
                f'Signal too short for max window '
                f'{MAX_REQUIRED_WINDOW}: {p}'
            )

        records.append(
            dict(
                rpm=rpm,
                label=label,
                class_name=name,
                path=p,
                signal=sig,
                split=split
            )
        )


record_table=pd.DataFrame([
    {
        k:(
            v.name
            if k=='path'
            else v
        )
        for k,v in r.items()
        if k!='signal'
    } |
    {
        'points':len(r['signal'])
    }
    for r in records
])

display(record_table)

print(
    'Source recordings:',
    len(records)
)

# Report the original, unscaled CWRU amplitude range.
TRAIN_MAX=max(
    float(
        np.max(
            np.abs(
                r['signal'][:r['split']]
            )
        )
    )
    for r in records
)

ALL_MIN=min(
    float(np.min(r['signal']))
    for r in records
)

ALL_MAX=max(
    float(np.max(r['signal']))
    for r in records
)

# With SAMPLE_INT_BITS=4, the smallest format is Q4.0.
# It represents integer values -8 through +7. Higher precisions approach
# the same nominal upper limit (+8) while adding fractional resolution.
Q4_0_MIN=-8.0
Q4_0_MAX=7.0

# No amplitude normalization is applied anywhere in the raw-sample path.
RAW_TO_FIXED_SCALE=1.0

# One common zero-crossing threshold in ORIGINAL CWRU amplitude units.
# 0.0625 is exactly one LSB for the 8-bit Q4.4 representation.
ZC_EPSILON_REAL=1.0/16.0

print('Train-only maximum absolute signal:',TRAIN_MAX)
print('Full-dataset signal range:',(ALL_MIN,ALL_MAX))
print('Smallest raw format range (Q4.0):',(Q4_0_MIN,Q4_0_MAX))
print('Raw amplitude scale:',RAW_TO_FIXED_SCALE,'(no scaling)')
print('Common ZC epsilon:',ZC_EPSILON_REAL)

if ALL_MIN<Q4_0_MIN or ALL_MAX>Q4_0_MAX:
    print(
        'WARNING: some samples are outside the Q4.0 range and will saturate '
        'only in the 4-bit experiment. The raw signal is still unscaled.'
    )
else:
    print(
        'All dataset samples fit inside Q4.0; no raw-sample saturation is '
        'required for any precision.'
    )


,rpm,label,class_name,path,split,points
0,1797,0,Normal,1797_Normal.npz,42689,60985
1,1797,1,B007,1797_B_7_DE12.npz,85799,122571
2,1797,2,IR007,1797_IR_7_DE12.npz,84885,121265
3,1797,3,OR007,1797_OR@6_7_DE12.npz,85393,121991
4,1797,4,B014,1797_B_14_DE12.npz,85292,121846
5,1797,5,IR014,1797_IR_14_DE12.npz,85292,121846
6,1797,6,OR014,1797_OR@6_14_DE12.npz,85292,121846
7,1797,7,B021,1797_B_21_DE12.npz,85393,121991
8,1797,8,IR021,1797_IR_21_DE12.npz,85495,122136
9,1797,9,OR021,1797_OR@6_21_DE12.npz,85698,122426


Source recordings: 40
Train-only maximum absolute signal: 6.653342914171657
Full-dataset signal range: (-6.653342914171657, 6.652936826347305)
Smallest raw format range (Q4.0): (-8.0, 7.0)
Raw amplitude scale: 1.0 (no scaling)
Common ZC epsilon: 0.0625
All dataset samples fit inside Q4.0; no raw-sample saturation is required for any precision.


## Helper functions


In [5]:
def geometry(window_size,raw_sample_bits):
    samples_per_bus=(
        BUS_BITS//raw_sample_bits
    )

    samples_per_packet=(
        NUM_INPUT_BUSES*
        samples_per_bus
    )

    packets_per_window=(
        window_size+
        samples_per_packet-1
    )//samples_per_packet

    last_packet_valid_samples=(
        window_size-
        (packets_per_window-1)*
        samples_per_packet
    )

    return {
        'samples_per_bus':samples_per_bus,
        'samples_per_packet':samples_per_packet,
        'packets_per_window':packets_per_window,
        'last_packet_valid_samples':last_packet_valid_samples,
        'bus_bits_used':(
            samples_per_bus*
            raw_sample_bits
        ),
    }


def required_signed_int_bits(values):
    """
    Smallest ap_fixed integer width I, including the sign bit,
    required to represent all supplied values.
    """
    m=float(
        np.max(
            np.abs(
                np.asarray(
                    values,
                    dtype=np.float64
                )
            )
        )
    )

    if m==0.0:
        return 2

    return max(
        2,
        int(
            np.ceil(
                np.log2(
                    m+1e-12
                )
            )
        )+1
    )


def quantize_signed_fixed(
    x,
    total_bits,
    int_bits
):
    frac_bits=(
        total_bits-
        int_bits
    )

    scale=2**frac_bits

    q=np.rint(
        np.asarray(
            x,
            dtype=np.float64
        )*scale
    )

    q=np.clip(
        q,
        -(2**(total_bits-1)),
        2**(total_bits-1)-1
    )

    return q/scale


def choose_starts(
    lo,
    hi,
    count,
    window_size,
    rng
):
    cand=np.arange(
        lo,
        hi-window_size+1,
        STRIDE,
        dtype=np.int64
    )

    if len(cand)<count:
        raise ValueError(
            f'{len(cand)} candidates '
            f'< {count}'
        )

    out=rng.choice(
        cand,
        size=count,
        replace=False
    )

    out.sort()

    return out


def class_metric_table(
    y_true,
    y_pred
):
    precision=precision_score(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        average=None,
        zero_division=0
    )

    recall=recall_score(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        average=None,
        zero_division=0
    )

    f1=f1_score(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        average=None,
        zero_division=0
    )

    support=np.bincount(
        np.asarray(
            y_true,
            dtype=np.int32
        ),
        minlength=NUM_CLASSES
    )

    return pd.DataFrame({
        'class_id':np.arange(NUM_CLASSES),
        'class_name':CLASS_NAMES,
        'precision':precision,
        'recall':recall,
        'f1':f1,
        'support':support,
    })


## Train and export one configuration


In [6]:
def run_experiment(
    window_size,
    raw_sample_bits
):
    assert (
        window_size
        in SUPPORTED_WINDOW_SIZES
    )

    assert (
        raw_sample_bits
        in SUPPORTED_RAW_BITS
    )

    cfg=(
        f'w{window_size}_'
        f'b{raw_sample_bits}'
    )

    print()
    print('='*78)
    print(
        'TRAINING',
        cfg,
        f'({window_size} samples, '
        f'{raw_sample_bits}-bit raw)'
    )
    print('='*78)

    # --------------------------------------------------------
    # Configuration-dependent numerical / packet parameters.
    # --------------------------------------------------------

    g=geometry(
        window_size,
        raw_sample_bits
    )

    samples_per_bus=(
        g['samples_per_bus']
    )

    samples_per_packet=(
        g['samples_per_packet']
    )

    packets_per_window=(
        g['packets_per_window']
    )

    last_packet_valid_samples=(
        g[
            'last_packet_valid_samples'
        ]
    )

    bus_bits_used=(
        g['bus_bits_used']
    )

    sample_int_bits=SAMPLE_INT_BITS
    sample_frac_bits=(
        raw_sample_bits-
        sample_int_bits
    )

    assert sample_frac_bits>=0

    raw_code_scale=(
        1 << sample_frac_bits
    )

    raw_code_min=(
        -(1 << (
            raw_sample_bits-1
        ))
    )

    raw_code_max=(
        (1 << (
            raw_sample_bits-1
        ))-1
    )

    raw_fixed_min=(
        raw_code_min/
        raw_code_scale
    )

    raw_fixed_max=(
        raw_code_max/
        raw_code_scale
    )

    # No raw-amplitude normalization. The original CWRU numerical value is
    # quantized directly into Q4.x. Only the number of fractional bits changes.
    raw_to_fixed_scale=RAW_TO_FIXED_SCALE

    # Quantize the same numerical ZC threshold into this raw sample format.
    zc_epsilon_code=int(
        np.clip(
            np.rint(
                ZC_EPSILON_REAL*
                raw_code_scale
            ),
            0,
            raw_code_max
        )
    )

    zc_epsilon=(
        zc_epsilon_code/
        raw_code_scale
    )

    feature_int_bits=(
        FEATURE_INT_BITS
    )

    # Preserve the raw sample fractional resolution through Peak/MAV and
    # into the MLR feature representation. This avoids making the feature
    # type itself the bottleneck for the 12- and 16-bit experiments.
    feature_frac_bits=(
        sample_frac_bits
    )

    feature_total_bits=(
        feature_int_bits+
        feature_frac_bits
    )

    # --------------------------------------------------------
    # Quantizer and three hardware-matched features.
    # --------------------------------------------------------

    def qsignal(sig):
        """
        Quantize the ORIGINAL, unscaled CWRU vibration amplitude directly
        into raw ap_fixed bit codes.

        The formats are Q4.0/Q4.4/Q4.8/Q4.12. No physical normalization
        multiplier is applied; increasing SAMPLE_BITS only adds fractional
        resolution. Values outside a format's representable range saturate
        exactly as AP_SAT does in HLS.
        """
        values=np.asarray(
            sig,
            dtype=np.float64
        )

        codes=np.rint(
            values*
            raw_code_scale
        )

        codes=np.clip(
            codes,
            raw_code_min,
            raw_code_max
        )

        return codes.astype(
            np.int16
        )


    def quantize_mav_feature(
        abs_code_sum
    ):
        """
        Hardware-matched fixed-point MAV.

        The accumulator and raw samples share the same binary-point position.
        Division by a power-of-two window is therefore just >> log2(N).
        The result keeps sample_frac_bits fractional bits naturally; there is
        no '(shift - FEATURE_FRAC_BITS)' compensation anymore.
        """
        shift=int(
            np.log2(
                window_size
            )
        )

        assert (
            (1 << shift)
            == window_size
        )

        mav_code=(
            np.asarray(
                abs_code_sum,
                dtype=np.int64
            ) >>
            shift
        )

        return (
            mav_code.astype(
                np.float64
            )/
            raw_code_scale
        )


    def features(
        sig,
        starts
    ):
        out=np.empty(
            (
                len(starts),
                NUM_FEATURES
            ),
            dtype=np.float32
        )

        for row,s in enumerate(starts):
            # qsignal() stores the signed raw fixed-point bit codes.
            w_code=np.asarray(
                sig[
                    s:
                    s+window_size
                ],
                dtype=np.int32
            )

            # Peak preserves the raw fixed-point fractional resolution.
            peak=float(
                np.max(w_code)/
                raw_code_scale
            )

            # Sum absolute fixed-point codes, then divide by N with the normal
            # power-of-two right shift. Converting the resulting code back by
            # raw_code_scale preserves the ap_fixed fractional bits.
            mav_sum_code=np.sum(
                np.abs(
                    w_code.astype(
                        np.int64
                    )
                ),
                dtype=np.int64
            )

            mav=float(
                quantize_mav_feature(
                    mav_sum_code
                )
            )

            a=w_code[:-1]
            b=w_code[1:]

            sign=(
                ((a<0)&(b>0)) |
                ((a>0)&(b<0))
            )

            delta_code=np.abs(
                b.astype(np.int64)-
                a.astype(np.int64)
            )

            zc=float(
                np.count_nonzero(
                    sign &
                    (
                        delta_code>
                        zc_epsilon_code
                    )
                )
            )

            out[row]=[
                peak,
                mav,
                zc
            ]

        return out

    # --------------------------------------------------------
    # Build balanced train/test sets.
    #
    # Resetting the RNG for every configuration preserves
    # reproducibility. Configurations with the same window size
    # therefore use the same selected window start positions.
    # --------------------------------------------------------

    rng=np.random.default_rng(
        RANDOM_STATE
    )

    Xtr=[]
    ytr=[]
    Xte=[]
    yte=[]
    Wte=[]

    train_meta=[]
    test_meta=[]

    for r in records:
        q=qsignal(
            r['signal']
        )

        ts=choose_starts(
            0,
            r['split'],
            TRAIN_WINDOWS_PER_RECORD,
            window_size,
            rng
        )

        vs=choose_starts(
            r['split'],
            len(q),
            TEST_WINDOWS_PER_RECORD,
            window_size,
            rng
        )

        Xtr.append(
            features(
                q,
                ts
            )
        )

        Xte.append(
            features(
                q,
                vs
            )
        )

        Wte.append(
            np.stack([
                q[
                    s:
                    s+window_size
                ]
                for s in vs
            ]).astype(
                np.int16
            )
        )

        ytr.append(
            np.full(
                len(ts),
                r['label'],
                np.int32
            )
        )

        yte.append(
            np.full(
                len(vs),
                r['label'],
                np.int32
            )
        )

        train_meta += [
            dict(
                rpm=r['rpm'],
                label=r['label'],
                class_name=r['class_name'],
                file=r['path'].name,
                start=int(s)
            )
            for s in ts
        ]

        test_meta += [
            dict(
                rpm=r['rpm'],
                label=r['label'],
                class_name=r['class_name'],
                file=r['path'].name,
                start=int(s)
            )
            for s in vs
        ]

    X_train_raw=np.concatenate(
        Xtr
    )

    y_train=np.concatenate(
        ytr
    )

    X_test_raw=np.concatenate(
        Xte
    )

    y_test=np.concatenate(
        yte
    )

    X_test_windows=np.concatenate(
        Wte
    )

    train_meta=pd.DataFrame(
        train_meta
    )

    test_meta=pd.DataFrame(
        test_meta
    )

    # Preserve the same shuffled row order behavior.
    pt=rng.permutation(
        len(y_train)
    )

    pv=rng.permutation(
        len(y_test)
    )

    X_train_raw=(
        X_train_raw[pt]
    )

    y_train=(
        y_train[pt]
    )

    train_meta=(
        train_meta
        .iloc[pt]
        .reset_index(
            drop=True
        )
    )

    X_test_raw=(
        X_test_raw[pv]
    )

    y_test=(
        y_test[pv]
    )

    X_test_windows=(
        X_test_windows[pv]
    )

    test_meta=(
        test_meta
        .iloc[pv]
        .reset_index(
            drop=True
        )
    )

    # --------------------------------------------------------
    # MLR hyperparameter search.
    # --------------------------------------------------------

    mlr_pipeline=Pipeline([
        (
            'scaler',
            StandardScaler()
        ),
        (
            'mlr',
            LogisticRegression(
                solver='lbfgs',
                max_iter=5000,
                random_state=RANDOM_STATE
            )
        )
    ])

    param_grid={
        'mlr__C':C_VALUES
    }

    if PROTOCOL=='all_loads':
        cv=StratifiedGroupKFold(
            n_splits=4,
            shuffle=True,
            random_state=RANDOM_STATE
        )

        cv_groups=(
            train_meta[
                'file'
            ].to_numpy()
        )

        grid=GridSearchCV(
            estimator=mlr_pipeline,
            param_grid=param_grid,
            scoring='balanced_accuracy',
            cv=cv,
            n_jobs=-1,
            refit=True,
            return_train_score=True
        )

        tuning_start=(
            time.perf_counter()
        )

        grid.fit(
            X_train_raw,
            y_train,
            groups=cv_groups
        )

    else:
        cv=StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=RANDOM_STATE
        )

        grid=GridSearchCV(
            estimator=mlr_pipeline,
            param_grid=param_grid,
            scoring='balanced_accuracy',
            cv=cv,
            n_jobs=-1,
            refit=True,
            return_train_score=True
        )

        tuning_start=(
            time.perf_counter()
        )

        grid.fit(
            X_train_raw,
            y_train
        )

    tuning_time=(
        time.perf_counter()-
        tuning_start
    )

    best_pipeline=(
        grid.best_estimator_
    )

    scaler=(
        best_pipeline
        .named_steps['scaler']
    )

    model=(
        best_pipeline
        .named_steps['mlr']
    )

    X_train=(
        scaler.transform(
            X_train_raw
        ).astype(
            np.float32
        )
    )

    X_test=(
        scaler.transform(
            X_test_raw
        ).astype(
            np.float32
        )
    )

    pred=(
        model.predict(
            X_test
        ).astype(
            np.int32
        )
    )

    # --------------------------------------------------------
    # Fold StandardScaler into hardware coefficients.
    # --------------------------------------------------------

    W=(
        model.coef_
        .astype(
            np.float64
        )
    )

    B=(
        model.intercept_
        .astype(
            np.float64
        )
    )

    manual_logits=(
        X_test.astype(
            np.float64
        ) @ W.T + B
    )

    manual_pred=np.argmax(
        manual_logits,
        axis=1
    ).astype(
        np.int32
    )

    W_hw=(
        W/
        scaler.scale_[
            None,
            :
        ]
    )

    B_hw=(
        B-
        W_hw @
        scaler.mean_
    )

    folded_logits=(
        X_test_raw.astype(
            np.float64
        ) @
        W_hw.T+
        B_hw
    )

    folded_pred=np.argmax(
        folded_logits,
        axis=1
    ).astype(
        np.int32
    )

    # --------------------------------------------------------
    # 18-bit folded-weight quantization, PER FEATURE COLUMN.
    #
    # Peak, MAV, and ZC live in very different numerical units after
    # StandardScaler is folded into the coefficients. A single Q-format
    # for all 30 weights can be dominated by one large Peak/MAV weight and
    # leave only a few fractional bits for the much smaller ZC weights.
    # Each feature column therefore receives its own 18-bit Q-format.
    # This changes representation only; the MLR equation is unchanged.
    # --------------------------------------------------------

    weight_int_bits=np.array([
        min(
            required_signed_int_bits(
                W_hw[:,f]
            ),
            WEIGHT_TOTAL_BITS
        )
        for f in range(NUM_FEATURES)
    ],dtype=np.int32)

    weight_frac_bits=(
        WEIGHT_TOTAL_BITS-
        weight_int_bits
    ).astype(np.int32)

    bias_int_bits=min(
        required_signed_int_bits(
            B_hw
        ),
        BIAS_TOTAL_BITS
    )

    bias_frac_bits=(
        BIAS_TOTAL_BITS-
        bias_int_bits
    )

    W_hw_q18=np.empty_like(
        W_hw,
        dtype=np.float64
    )

    for f in range(NUM_FEATURES):
        W_hw_q18[:,f]=quantize_signed_fixed(
            W_hw[:,f],
            WEIGHT_TOTAL_BITS,
            int(weight_int_bits[f])
        )

    B_hw_q18=quantize_signed_fixed(
        B_hw,
        BIAS_TOTAL_BITS,
        bias_int_bits
    )

    logits_q18=(
        X_test_raw.astype(
            np.float64
        ) @
        W_hw_q18.T+
        B_hw_q18
    )

    pred_q18=np.argmax(
        logits_q18,
        axis=1
    ).astype(
        np.int32
    )

    # --------------------------------------------------------
    # Requested accuracy / precision / recall / F1 metrics.
    # --------------------------------------------------------

    q18_accuracy=accuracy_score(
        y_test,
        pred_q18
    )

    q18_balanced_accuracy=(
        balanced_accuracy_score(
            y_test,
            pred_q18
        )
    )

    q18_macro_precision=precision_score(
        y_test,
        pred_q18,
        average='macro',
        zero_division=0
    )

    q18_macro_recall=recall_score(
        y_test,
        pred_q18,
        average='macro',
        zero_division=0
    )

    q18_macro_f1=f1_score(
        y_test,
        pred_q18,
        average='macro',
        zero_division=0
    )

    q18_weighted_precision=precision_score(
        y_test,
        pred_q18,
        average='weighted',
        zero_division=0
    )

    q18_weighted_recall=recall_score(
        y_test,
        pred_q18,
        average='weighted',
        zero_division=0
    )

    q18_weighted_f1=f1_score(
        y_test,
        pred_q18,
        average='weighted',
        zero_division=0
    )

    fault_true=(
        y_test!=0
    )

    fault_pred=(
        pred_q18!=0
    )

    q18_fault_precision=precision_score(
        fault_true,
        fault_pred,
        zero_division=0
    )

    q18_fault_recall=recall_score(
        fault_true,
        fault_pred,
        zero_division=0
    )

    q18_fault_f1=f1_score(
        fault_true,
        fault_pred,
        zero_division=0
    )

    # --------------------------------------------------------
    # Numerical range checks matching HLS feature_t.
    # --------------------------------------------------------

    feature_max=(
        2**(
            feature_int_bits-1
        )-
        2**(
            -feature_frac_bits
        )
    )

    feature_min=(
        -(
            2**(
                feature_int_bits-1
            )
        )
    )

    assert (
        X_train_raw.max()
        <= feature_max
    )

    assert (
        X_train_raw.min()
        >= feature_min
    )

    # --------------------------------------------------------
    # Save normal per-experiment archive.
    # --------------------------------------------------------

    OUT=Path(
        f'cwru_mlr_output_'
        f'{PROTOCOL}_'
        f'w{window_size}_'
        f'raw{raw_sample_bits}b'
    )

    OUT.mkdir(
        exist_ok=True
    )

    np.save(
        OUT/
        'X_test_mlr_quantized_windows.npy',
        X_test_windows.astype(
            np.int16
        )
    )

    np.save(
        OUT/
        'X_test_mlr_raw_features.npy',
        X_test_raw.astype(
            np.float32
        )
    )

    np.save(
        OUT/
        'X_test_mlr_scaled.npy',
        X_test.astype(
            np.float32
        )
    )

    np.save(
        OUT/
        'y_test_mlr.npy',
        y_test.astype(
            np.int32
        )
    )

    train_meta.to_csv(
        OUT/
        'train_metadata.csv',
        index=False
    )

    test_meta.to_csv(
        OUT/
        'test_metadata.csv',
        index=False
    )

    cv_results=(
        pd.DataFrame(
            grid.cv_results_
        )[[
            'param_mlr__C',
            'mean_train_score',
            'mean_test_score',
            'std_test_score'
        ]]
        .rename(
            columns={
                'param_mlr__C':'C',
                'mean_train_score':
                    'Train Balanced Accuracy',
                'mean_test_score':
                    'CV Balanced Accuracy',
                'std_test_score':
                    'CV Std',
            }
        )
    )

    cv_results.to_csv(
        OUT/
        'cv_results.csv',
        index=False
    )

    per_class=class_metric_table(
        y_test,
        pred_q18
    )

    per_class.to_csv(
        OUT/
        'q18_per_class_metrics.csv',
        index=False
    )

    cm_q18=confusion_matrix(
        y_test,
        pred_q18,
        labels=np.arange(
            NUM_CLASSES
        )
    )

    pd.DataFrame(
        cm_q18,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    ).to_csv(
        OUT/
        'q18_confusion_matrix.csv'
    )

    np.savez(
        OUT/
        'mlr_params.npz',
        mean=scaler.mean_.astype(
            np.float32
        ),
        scale=scaler.scale_.astype(
            np.float32
        ),
        weights=model.coef_.astype(
            np.float32
        ),
        bias=model.intercept_.astype(
            np.float32
        ),
        hw_weights=W_hw.astype(
            np.float32
        ),
        hw_bias=B_hw.astype(
            np.float32
        ),
        hw_weights_q18=W_hw_q18.astype(
            np.float32
        ),
        hw_bias_q18=B_hw_q18.astype(
            np.float32
        ),
        classes=model.classes_.astype(
            np.int32
        ),
        class_names=np.array(
            CLASS_NAMES
        ),
        feature_names=np.array(
            FEATURE_NAMES
        ),
        rpm_list=np.array(
            RPM_LIST,
            np.int32
        ),
        window_size=np.array(
            [window_size],
            np.int32
        ),
        stride=np.array(
            [STRIDE],
            np.int32
        ),
        raw_sample_bits=np.array(
            [raw_sample_bits],
            np.int32
        ),
        bus_bits=np.array(
            [BUS_BITS],
            np.int32
        ),
        num_input_buses=np.array(
            [NUM_INPUT_BUSES],
            np.int32
        ),
        samples_per_bus=np.array(
            [samples_per_bus],
            np.int32
        ),
        samples_per_packet=np.array(
            [samples_per_packet],
            np.int32
        ),
        packets_per_window=np.array(
            [packets_per_window],
            np.int32
        ),
        last_packet_valid_samples=np.array(
            [
                last_packet_valid_samples
            ],
            np.int32
        ),
        raw_fixed_scale=np.array(
            [raw_to_fixed_scale],
            np.float32
        ),
        sample_int_bits=np.array(
            [sample_int_bits],
            np.int32
        ),
        sample_frac_bits=np.array(
            [sample_frac_bits],
            np.int32
        ),
        raw_code_scale=np.array(
            [raw_code_scale],
            np.int32
        ),
        zc_epsilon_code=np.array(
            [zc_epsilon_code],
            np.int32
        ),
        zc_epsilon=np.array(
            [zc_epsilon],
            np.float32
        ),
        feature_total_bits=np.array(
            [feature_total_bits],
            np.int32
        ),
        feature_int_bits=np.array(
            [feature_int_bits],
            np.int32
        ),
        weight_total_bits=np.array(
            [WEIGHT_TOTAL_BITS],
            np.int32
        ),
        weight_int_bits=np.asarray(
            weight_int_bits,
            dtype=np.int32
        ),
        bias_total_bits=np.array(
            [BIAS_TOTAL_BITS],
            np.int32
        ),
        bias_int_bits=np.array(
            [bias_int_bits],
            np.int32
        )
    )

    summary={
        'config':cfg,
        'window_size':window_size,
        'raw_sample_bits':raw_sample_bits,
        'sample_int_bits':sample_int_bits,
        'sample_frac_bits':sample_frac_bits,
        'raw_fixed_scale':raw_to_fixed_scale,
        'samples_per_bus':samples_per_bus,
        'samples_per_packet':samples_per_packet,
        'packets_per_window':packets_per_window,
        'last_packet_valid_samples':
            last_packet_valid_samples,
        'bus_bits_used':bus_bits_used,
        'zc_epsilon_code':zc_epsilon_code,
        'zc_epsilon':zc_epsilon,
        'feature_int_bits':
            feature_int_bits,
        'feature_frac_bits':
            feature_frac_bits,
        'peak_weight_int_bits':
            int(weight_int_bits[0]),
        'peak_weight_frac_bits':
            int(weight_frac_bits[0]),
        'mav_weight_int_bits':
            int(weight_int_bits[1]),
        'mav_weight_frac_bits':
            int(weight_frac_bits[1]),
        'zc_weight_int_bits':
            int(weight_int_bits[2]),
        'zc_weight_frac_bits':
            int(weight_frac_bits[2]),
        'bias_int_bits':
            bias_int_bits,
        'bias_frac_bits':
            bias_frac_bits,
        'best_C':float(
            grid.best_params_[
                'mlr__C'
            ]
        ),
        'cv_balanced_accuracy':
            float(
                grid.best_score_
            ),
        'float_accuracy':
            float(
                accuracy_score(
                    y_test,
                    folded_pred
                )
            ),
        'q18_accuracy':
            float(
                q18_accuracy
            ),
        'q18_balanced_accuracy':
            float(
                q18_balanced_accuracy
            ),
        'q18_macro_precision':
            float(
                q18_macro_precision
            ),
        'q18_macro_recall':
            float(
                q18_macro_recall
            ),
        'q18_macro_f1':
            float(
                q18_macro_f1
            ),
        'q18_weighted_precision':
            float(
                q18_weighted_precision
            ),
        'q18_weighted_recall':
            float(
                q18_weighted_recall
            ),
        'q18_weighted_f1':
            float(
                q18_weighted_f1
            ),
        'q18_fault_precision':
            float(
                q18_fault_precision
            ),
        'q18_fault_recall':
            float(
                q18_fault_recall
            ),
        'q18_fault_f1':
            float(
                q18_fault_f1
            ),
        'q18_float_agreement':
            float(
                np.mean(
                    pred_q18==
                    folded_pred
                )
            ),
        'manual_sklearn_agreement':
            float(
                np.mean(
                    manual_pred==
                    pred
                )
            ),
        'tuning_time_s':
            float(
                tuning_time
            ),
    }

    pd.DataFrame(
        [summary]
    ).to_csv(
        OUT/
        'model_precision_summary.csv',
        index=False
    )

    float_accuracy=float(
        accuracy_score(
            y_test,
            folded_pred
        )
    )

    q18_float_agreement=float(
        np.mean(
            pred_q18==folded_pred
        )
    )

    print(
        f'{cfg}: '
        f'float acc={float_accuracy:.6f}, '
        f'Q18-per-feature acc={q18_accuracy:.6f}, '
        f'agreement={q18_float_agreement:.6f}, '
        f'macro F1={q18_macro_f1:.6f}, '
        f'fault recall={q18_fault_recall:.6f}, '
        f'best C={grid.best_params_["mlr__C"]}'
    )

    print(
        f'  packing: '
        f'{samples_per_bus} samples/bus, '
        f'{samples_per_packet} samples/packet, '
        f'{packets_per_window} packets/window'
    )

    print(
        f'  formats: '
        f'raw Q{sample_int_bits}.'
        f'{sample_frac_bits}, '
        f'feature Q{feature_int_bits}.'
        f'{feature_frac_bits}, '
        f'weights Peak Q{int(weight_int_bits[0])}.'
        f'{int(weight_frac_bits[0])}, '
        f'MAV Q{int(weight_int_bits[1])}.'
        f'{int(weight_frac_bits[1])}, '
        f'ZC Q{int(weight_int_bits[2])}.'
        f'{int(weight_frac_bits[2])}, '
        f'bias Q{bias_int_bits}.'
        f'{bias_frac_bits}'
    )

    return summary


## Run all 16 experiments


In [7]:
sweep_start=time.perf_counter()

sweep_summaries=[]

for window_size in SUPPORTED_WINDOW_SIZES:
    for raw_sample_bits in SUPPORTED_RAW_BITS:
        summary=run_experiment(
            window_size,
            raw_sample_bits
        )

        sweep_summaries.append(
            summary
        )

sweep_time=(
    time.perf_counter()-
    sweep_start
)

sweep_summary=pd.DataFrame(
    sweep_summaries
)

sweep_summary.to_csv(
    'sweep_model_summary.csv',
    index=False
)

print()
print('='*78)
print('FULL 4x4 SWEEP COMPLETE')
print('='*78)

print(
    'Configurations trained:',
    len(sweep_summary)
)

print(
    'Total sweep time (s):',
    sweep_time
)

display(
    sweep_summary
)



TRAINING w256_b4 (256 samples, 4-bit raw)
w256_b4: float acc=0.361833, Q18-per-feature acc=0.361833, agreement=1.000000, macro F1=0.280904, fault recall=0.559074, best C=500.0
  packing: 16 samples/bus, 64 samples/packet, 4 packets/window
  formats: raw Q4.0, feature Q12.0, weights Peak Q5.13, MAV Q2.16, ZC Q3.15, bias Q4.14

TRAINING w256_b8 (256 samples, 8-bit raw)
w256_b8: float acc=0.687333, Q18-per-feature acc=0.687333, agreement=1.000000, macro F1=0.682659, fault recall=0.981667, best C=300.0
  packing: 8 samples/bus, 32 samples/packet, 8 packets/window
  formats: raw Q4.4, feature Q12.4, weights Peak Q7.11, MAV Q9.9, ZC Q2.16, bias Q6.12

TRAINING w256_b12 (256 samples, 12-bit raw)
w256_b12: float acc=0.768000, Q18-per-feature acc=0.768167, agreement=0.999833, macro F1=0.765077, fault recall=0.995000, best C=150.0
  packing: 5 samples/bus, 20 samples/packet, 13 packets/window
  formats: raw Q4.8, feature Q12.8, weights Peak Q6.12, MAV Q11.7, ZC Q2.16, bias Q7.11

TRAINING w256_

,config,window_size,raw_sample_bits,sample_int_bits,sample_frac_bits,raw_fixed_scale,samples_per_bus,samples_per_packet,packets_per_window,last_packet_valid_samples,...,q18_macro_f1,q18_weighted_precision,q18_weighted_recall,q18_weighted_f1,q18_fault_precision,q18_fault_recall,q18_fault_f1,q18_float_agreement,manual_sklearn_agreement,tuning_time_s
0,w256_b4,256,4,4,0,1.0,16,64,4,64,...,0.280904,0.271040,0.361833,0.280904,1.000000,0.559074,0.717187,1.000000,1.0,11.044675
1,w256_b8,256,8,4,4,1.0,8,32,8,32,...,0.682659,0.684037,0.687333,0.682659,0.995867,0.981667,0.988716,1.000000,1.0,4.166956
2,w256_b12,256,12,4,8,1.0,5,20,13,16,...,0.765077,0.767772,0.768167,0.765077,0.995553,0.995000,0.995276,0.999833,1.0,4.495126
3,w256_b16,256,16,4,12,1.0,4,16,16,16,...,0.765555,0.768604,0.768500,0.765555,0.994094,0.997407,0.995748,1.000000,1.0,4.558840
4,w512_b4,512,4,4,0,1.0,16,64,8,64,...,0.412483,0.414578,0.469500,0.412483,1.000000,0.614444,0.761184,1.000000,1.0,2.892078
5,w512_b8,512,8,4,4,1.0,8,32,16,32,...,0.825133,0.826744,0.825500,0.825133,0.997772,0.995000,0.996384,1.000000,1.0,5.587759
6,w512_b12,512,12,4,8,1.0,5,20,26,12,...,0.888308,0.888884,0.889167,0.888308,0.999074,0.999444,0.999259,0.999833,1.0,4.622449
7,w512_b16,512,16,4,12,1.0,4,16,32,16,...,0.892995,0.893682,0.893833,0.892995,0.999075,0.999815,0.999445,0.999833,1.0,4.856916
8,w1024_b4,1024,4,4,0,1.0,16,64,16,64,...,0.510867,0.499179,0.572167,0.510867,1.000000,0.666481,0.799867,1.000000,1.0,4.550719
9,w1024_b8,1024,8,4,4,1.0,8,32,32,32,...,0.912369,0.913502,0.912667,0.912369,1.000000,0.998148,0.999073,1.000000,1.0,5.391234


## Build one master `logistic_params.h`


In [8]:
MASTER_HEADER=Path(
    'logistic_params.h'
)


def ff(v):
    return f'{float(v):.10f}'


def model_macro(
    window_size,
    raw_bits
):
    return (
        f'MODEL_W'
        f'{int(window_size)}_'
        f'B{int(raw_bits)}'
    )


def scalar_int(
    npz,
    key
):
    return int(
        np.asarray(
            npz[key]
        ).reshape(-1)[0]
    )


def model_npz_path(
    window_size,
    raw_bits
):
    return Path(
        f'cwru_mlr_output_'
        f'{PROTOCOL}_'
        f'w{int(window_size)}_'
        f'raw{int(raw_bits)}b'
    )/'mlr_params.npz'


def load_saved_model(
    window_size,
    raw_bits
):
    path=model_npz_path(
        window_size,
        raw_bits
    )

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    q=np.load(
        path,
        allow_pickle=False
    )

    entry={
        'window_size':
            scalar_int(
                q,
                'window_size'
            ),
        'raw_sample_bits':
            scalar_int(
                q,
                'raw_sample_bits'
            ),
        'sample_int_bits':
            scalar_int(
                q,
                'sample_int_bits'
            ),
        'sample_frac_bits':
            scalar_int(
                q,
                'sample_frac_bits'
            ),
        'zc_epsilon_code':
            scalar_int(
                q,
                'zc_epsilon_code'
            ),
        'feature_total_bits':
            scalar_int(
                q,
                'feature_total_bits'
            ),
        'feature_int_bits':
            scalar_int(
                q,
                'feature_int_bits'
            ),
        'weight_total_bits':
            scalar_int(
                q,
                'weight_total_bits'
            ),
        'weight_int_bits':
            np.asarray(
                q['weight_int_bits'],
                dtype=np.int32
            ).reshape(NUM_FEATURES),
        'bias_total_bits':
            scalar_int(
                q,
                'bias_total_bits'
            ),
        'bias_int_bits':
            scalar_int(
                q,
                'bias_int_bits'
            ),
        'weights':
            np.asarray(
                q['hw_weights_q18'],
                dtype=np.float64
            ),
        'bias':
            np.asarray(
                q['hw_bias_q18'],
                dtype=np.float64
            ),
    }

    assert (
        entry['window_size']
        == int(window_size)
    )

    assert (
        entry['raw_sample_bits']
        == int(raw_bits)
    )

    assert (
        entry['weights'].shape
        == (
            NUM_CLASSES,
            NUM_FEATURES
        )
    )

    assert (
        entry['bias'].shape
        == (
            NUM_CLASSES,
        )
    )

    return entry


def write_model_block(
    f,
    entry,
    first
):
    w=entry['window_size']
    b=entry['raw_sample_bits']

    macro=model_macro(
        w,
        b
    )

    directive=(
        '#if'
        if first
        else '#elif'
    )

    f.write(
        f'{directive} '
        f'defined({macro})\n\n'
    )

    feature_frac=(
        entry['feature_total_bits']-
        entry['feature_int_bits']
    )

    weight_frac=(
        entry['weight_total_bits']-
        entry['weight_int_bits']
    ).astype(np.int32)

    bias_frac=(
        entry['bias_total_bits']-
        entry['bias_int_bits']
    )

    f.write(
        f'#define SAMPLE_BITS {b}\n'
    )

    f.write(
        f'#define MODEL_WINDOW_SIZE {w}\n'
    )

    f.write(
        f'#define SAMPLE_INT_BITS '
        f'{entry["sample_int_bits"]}\n'
    )

    f.write(
        f'#define SAMPLE_FRAC_BITS '
        f'{entry["sample_frac_bits"]}\n'
    )

    f.write(
        f'#define ZC_EPSILON_CODE '
        f'{entry["zc_epsilon_code"]}\n\n'
    )

    f.write(
        f'#define FEATURE_TOTAL_BITS '
        f'{entry["feature_total_bits"]}\n'
    )

    f.write(
        f'#define FEATURE_INT_BITS '
        f'{entry["feature_int_bits"]}\n'
    )

    f.write(
        f'#define FEATURE_FRAC_BITS '
        f'{feature_frac}\n'
    )

    f.write(
        f'#define WEIGHT_TOTAL_BITS '
        f'{entry["weight_total_bits"]}\n'
    )

    for _name,_idx in [('PEAK',0),('MAV',1),('ZC',2)]:
        f.write(
            f'#define {_name}_WEIGHT_INT_BITS '
            f'{int(entry["weight_int_bits"][_idx])}\n'
        )
        f.write(
            f'#define {_name}_WEIGHT_FRAC_BITS '
            f'{int(weight_frac[_idx])}\n'
        )

    f.write('\n')

    f.write(
        f'#define BIAS_TOTAL_BITS '
        f'{entry["bias_total_bits"]}\n'
    )

    f.write(
        f'#define BIAS_INT_BITS '
        f'{entry["bias_int_bits"]}\n'
    )

    f.write(
        f'#define BIAS_FRAC_BITS '
        f'{bias_frac}\n\n'
    )

    f.write(
        'typedef ap_fixed<'
        'FEATURE_TOTAL_BITS, '
        'FEATURE_INT_BITS, '
        'AP_RND, AP_SAT> '
        'feature_t;\n'
    )

    f.write(
        'typedef ap_fixed<'
        'WEIGHT_TOTAL_BITS, '
        'PEAK_WEIGHT_INT_BITS, '
        'AP_RND, AP_SAT> '
        'peak_weight_t;\n'
    )
    f.write(
        'typedef ap_fixed<'
        'WEIGHT_TOTAL_BITS, '
        'MAV_WEIGHT_INT_BITS, '
        'AP_RND, AP_SAT> '
        'mav_weight_t;\n'
    )
    f.write(
        'typedef ap_fixed<'
        'WEIGHT_TOTAL_BITS, '
        'ZC_WEIGHT_INT_BITS, '
        'AP_RND, AP_SAT> '
        'zc_weight_t;\n'
    )

    f.write(
        'typedef ap_fixed<'
        'BIAS_TOTAL_BITS, '
        'BIAS_INT_BITS, '
        'AP_RND, AP_SAT> '
        'bias_t;\n\n'
    )

    _weight_arrays=[
        ('peak_weight_t','LR_PEAK_WEIGHTS',0),
        ('mav_weight_t','LR_MAV_WEIGHTS',1),
        ('zc_weight_t','LR_ZC_WEIGHTS',2),
    ]

    for _type,_array,_fidx in _weight_arrays:
        f.write(
            f'static const {_type} '
            f'{_array}[NUM_CLASSES] = {{'
        )
        f.write(
            ', '.join(
                ff(v)
                for v in entry['weights'][:,_fidx]
            )
        )
        f.write('};\n')

    f.write('\n')

    f.write(
        'static const bias_t '
        'LR_HW_BIAS'
        '[NUM_CLASSES] = {'
    )

    f.write(
        ', '.join(
            ff(v)
            for v in entry['bias']
        )
    )

    f.write(
        '};\n\n'
    )


all_entries=[]

for _w in SUPPORTED_WINDOW_SIZES:
    for _b in SUPPORTED_RAW_BITS:
        all_entries.append(
            load_saved_model(
                _w,
                _b
            )
        )


all_macros=[
    model_macro(
        _w,
        _b
    )
    for _w
    in SUPPORTED_WINDOW_SIZES
    for _b
    in SUPPORTED_RAW_BITS
]


with open(
    MASTER_HEADER,
    'w'
) as f:
    f.write(
        '#ifndef LOGISTIC_PARAMS_H\n'
    )

    f.write(
        '#define LOGISTIC_PARAMS_H\n\n'
    )

    f.write(
        '#include <ap_fixed.h>\n\n'
    )

    f.write(
        '// Master parameter file generated '
        'by the one-run full sweep notebook.\n'
    )

    f.write(
        '// Select exactly one '
        'MODEL_W<window>_B<bits> '
        'macro in logistic_fault.h.\n\n'
    )

    selector_expression=(
        ' + '.join(
            f'defined({m})'
            for m in all_macros
        )
    )
    
    f.write(
        f'#if ({selector_expression}) != 1\n'
    )
    
    f.write(
        '#error "Define exactly one '
        'MODEL_W<window>_B<bits> '
        'selector."\n'
    )
    
    f.write('#endif\n\n')

    f.write(
        f'#define NUM_FEATURES '
        f'{NUM_FEATURES}\n'
    )

    f.write(
        f'#define NUM_CLASSES '
        f'{NUM_CLASSES}\n\n'
    )

    for i,entry in enumerate(
        all_entries
    ):
        write_model_block(
            f,
            entry,
            first=(i==0)
        )

    f.write(
        '#else\n'
    )

    f.write(
        '#error "Selected model '
        'configuration not found."\n'
    )

    f.write(
        '#endif\n\n'
    )

    f.write(
        '#endif\n'
    )


print(
    'Created master HLS header:',
    MASTER_HEADER.resolve()
)

print(
    'Configurations in header:',
    len(all_entries)
)


Created master HLS header: C:\ECEN296\cwru_precision_sweep_64bit_single_header\logistic_params.h
Configurations in header: 16


## Build one master `experiment_data.npz`


In [9]:
MASTER_DATA=Path(
    'experiment_data.npz'
)


def config_key(
    window_size,
    raw_bits
):
    return (
        f'w{int(window_size)}_'
        f'b{int(raw_bits)}'
    )


def experiment_dir(
    window_size,
    raw_bits
):
    return Path(
        f'cwru_mlr_output_'
        f'{PROTOCOL}_'
        f'w{int(window_size)}_'
        f'raw{int(raw_bits)}b'
    )


def load_complete_experiment(
    window_size,
    raw_bits
):
    exp_dir=experiment_dir(
        window_size,
        raw_bits
    )

    paths={
        'windows':
            exp_dir/
            'X_test_mlr_quantized_windows.npy',
        'raw_features':
            exp_dir/
            'X_test_mlr_raw_features.npy',
        'scaled_features':
            exp_dir/
            'X_test_mlr_scaled.npy',
        'y_test':
            exp_dir/
            'y_test_mlr.npy',
        'params':
            exp_dir/
            'mlr_params.npz',
    }

    for name,path in paths.items():
        if not path.exists():
            raise FileNotFoundError(
                f'{name}: {path}'
            )

    q=np.load(
        paths['params'],
        allow_pickle=False
    )

    entry={
        # Signed raw ap_fixed bit codes, stored in int16 for convenient
        # packing into the SAMPLE_BITS-wide AXI lanes.
        'windows':
            np.load(
                paths['windows'],
                allow_pickle=False
            ).astype(
                np.int16
            ),
        'raw_features':
            np.load(
                paths['raw_features'],
                allow_pickle=False
            ).astype(
                np.float32
            ),
        'scaled_features':
            np.load(
                paths['scaled_features'],
                allow_pickle=False
            ).astype(
                np.float32
            ),
        'y_test':
            np.load(
                paths['y_test'],
                allow_pickle=False
            ).astype(
                np.int32
            ),
    }

    for key in q.files:
        entry[key]=np.asarray(
            q[key]
        )

    n=len(
        entry['y_test']
    )

    assert (
        entry['windows'].shape
        == (
            n,
            int(window_size)
        )
    )

    assert (
        entry['raw_features'].shape
        == (
            n,
            NUM_FEATURES
        )
    )

    assert (
        entry['scaled_features'].shape
        == (
            n,
            NUM_FEATURES
        )
    )

    assert (
        scalar_int(
            q,
            'window_size'
        )
        == int(window_size)
    )

    assert (
        scalar_int(
            q,
            'raw_sample_bits'
        )
        == int(raw_bits)
    )

    return entry


master_payload={}
available_configs=[]

for _w in SUPPORTED_WINDOW_SIZES:
    for _b in SUPPORTED_RAW_BITS:
        _cfg=config_key(
            _w,
            _b
        )

        _entry=load_complete_experiment(
            _w,
            _b
        )

        available_configs.append(
            _cfg
        )

        for field,value in _entry.items():
            master_payload[
                f'{_cfg}__{field}'
            ]=value


master_payload[
    'format_version'
]=np.array(
    [4],
    dtype=np.int32
)

master_payload[
    'available_configs'
]=np.asarray(
    available_configs,
    dtype='<U32'
)

master_payload[
    'supported_window_sizes'
]=np.asarray(
    SUPPORTED_WINDOW_SIZES,
    dtype=np.int32
)

master_payload[
    'supported_raw_bits'
]=np.asarray(
    SUPPORTED_RAW_BITS,
    dtype=np.int32
)


np.savez_compressed(
    MASTER_DATA,
    **master_payload
)


print(
    'Created master PYNQ data:',
    MASTER_DATA.resolve()
)

print(
    'Configurations in data archive:',
    len(available_configs)
)

print(
    'Available:',
    available_configs
)


Created master PYNQ data: C:\ECEN296\cwru_precision_sweep_64bit_single_header\experiment_data.npz
Configurations in data archive: 16
Available: ['w256_b4', 'w256_b8', 'w256_b12', 'w256_b16', 'w512_b4', 'w512_b8', 'w512_b12', 'w512_b16', 'w1024_b4', 'w1024_b8', 'w1024_b12', 'w1024_b16', 'w2048_b4', 'w2048_b8', 'w2048_b12', 'w2048_b16']


## Final verification


In [10]:
expected_configs=[
    (
        w,
        b
    )
    for w
    in SUPPORTED_WINDOW_SIZES
    for b
    in SUPPORTED_RAW_BITS
]

assert (
    len(expected_configs)
    == 16
)

missing=[]

for w,b in expected_configs:
    out=experiment_dir(
        w,
        b
    )

    required=[
        out/
        'mlr_params.npz',
        out/
        'X_test_mlr_quantized_windows.npy',
        out/
        'X_test_mlr_raw_features.npy',
        out/
        'X_test_mlr_scaled.npy',
        out/
        'y_test_mlr.npy',
        out/
        'model_precision_summary.csv',
        out/
        'q18_per_class_metrics.csv',
        out/
        'q18_confusion_matrix.csv',
    ]

    for path in required:
        if not path.exists():
            missing.append(
                str(path)
            )


if missing:
    raise FileNotFoundError(
        'Missing sweep outputs:\n' +
        '\n'.join(
            missing
        )
    )


assert Path(
    'logistic_params.h'
).exists()

assert Path(
    'experiment_data.npz'
).exists()

assert Path(
    'sweep_model_summary.csv'
).exists()


d=np.load(
    'experiment_data.npz',
    allow_pickle=False
)

assert (
    len(
        d[
            'available_configs'
        ]
    )
    == 16
)


print()
print('='*78)
print('ONE-RUN WORKFLOW COMPLETE')
print('='*78)
print(
    'All 16 configurations trained and verified.'
)
print(
    'Master HLS file:',
    Path(
        'logistic_params.h'
    ).resolve()
)
print(
    'Master PYNQ data:',
    Path(
        'experiment_data.npz'
    ).resolve()
)
print(
    'Sweep summary:',
    Path(
        'sweep_model_summary.csv'
    ).resolve()
)



ONE-RUN WORKFLOW COMPLETE
All 16 configurations trained and verified.
Master HLS file: C:\ECEN296\cwru_precision_sweep_64bit_single_header\logistic_params.h
Master PYNQ data: C:\ECEN296\cwru_precision_sweep_64bit_single_header\experiment_data.npz
Sweep summary: C:\ECEN296\cwru_precision_sweep_64bit_single_header\sweep_model_summary.csv


In [14]:
# ============================================================
# EXTRA FEATURE TEST
#
# Baseline: Peak + MAV + Zero Crossing
# Added:    Kurtosis + Normal Negative Log-Likelihood + Skewness


TEST_WINDOW_SIZE = 2048
TEST_RAW_BITS = 16

sample_frac_bits = TEST_RAW_BITS - SAMPLE_INT_BITS
raw_code_scale = 1 << sample_frac_bits

raw_code_min = -(1 << (TEST_RAW_BITS - 1))
raw_code_max =  (1 << (TEST_RAW_BITS - 1)) - 1

zc_epsilon_code = int(
    np.clip(
        np.rint(
            ZC_EPSILON_REAL * raw_code_scale
        ),
        0,
        raw_code_max
    )
)

mav_shift = int(
    np.log2(TEST_WINDOW_SIZE)
)

assert (1 << mav_shift) == TEST_WINDOW_SIZE


def qsignal_extra(sig):
    """
    Quantize original CWRU amplitudes into the same Q4.x raw codes
    used by the main training notebook.
    """
    q = np.rint(
        np.asarray(sig, dtype=np.float64)
        * raw_code_scale
    )

    q = np.clip(
        q,
        raw_code_min,
        raw_code_max
    )

    return q.astype(np.int16)


def extract_extra_features(sig_code, starts):
    """
    Returns:
      0 Peak
      1 MAV
      2 Zero Crossing
      3 Kurtosis
      4 Normal Negative Log-Likelihood
      5 Skewness
    """

    out = np.empty(
        (len(starts), 6),
        dtype=np.float64
    )

    for row, s in enumerate(starts):

        # Fixed-point integer codes for one window
        w_code = np.asarray(
            sig_code[
                s:s + TEST_WINDOW_SIZE
            ],
            dtype=np.int64
        )

        # Numerical Q4.x values
        w = (
            w_code.astype(np.float64)
            / raw_code_scale
        )

        # Existing feature 1: Peak
        peak = float(
            np.max(w_code)
            / raw_code_scale
        )

        # Existing feature 2: MAV
        mav_sum_code = np.sum(
            np.abs(w_code),
            dtype=np.int64
        )

        mav_code = (
            mav_sum_code
            >> mav_shift
        )

        mav = float(
            mav_code
            / raw_code_scale
        )

        # Existing feature 3: Zero Crossing
        a = w_code[:-1]
        b = w_code[1:]

        sign_change = (
            ((a < 0) & (b > 0)) |
            ((a > 0) & (b < 0))
        )

        delta_code = np.abs(
            b - a
        )

        zc = float(
            np.count_nonzero(
                sign_change &
                (
                    delta_code
                    > zc_epsilon_code
                )
            )
        )

        # ----------------------------------------------------
        # Statistics used by new features
        # ----------------------------------------------------
        mean = np.mean(w)

        centered = (
            w - mean
        )

        variance = np.mean(
            centered ** 2
        )

        # Constant windows would make skew/kurtosis/NLL undefined.
        if variance < 1e-12:
            skewness = 0.0
            kurtosis = 0.0
            normal_nll = 0.0

        else:
            std = np.sqrt(
                variance
            )
            
            # New feature 1: Skewness
            # Third standardized central moment.
            skewness = (
                np.mean(
                    centered ** 3
                )
                /
                (std ** 3)
            )

            # New feature 2: Kurtosis
            # Pearson kurtosis:
            # Gaussian distribution -> approximately 3.
            kurtosis = (
                np.mean(
                    centered ** 4
                )
                /
                (variance ** 2)
            )

            # New feature 3:
            # Normal Negative Log-Likelihood
            # Fit one Gaussian to this window using its
            # own mean and variance and compute -log L.
            normal_nll = (
                0.5
                * TEST_WINDOW_SIZE
                * np.log(
                    2.0
                    * np.pi
                    * variance
                )
                +
                np.sum(
                    centered ** 2
                )
                /
                (
                    2.0
                    * variance
                )
            )

        out[row] = [
            peak,
            mav,
            zc,
            kurtosis,
            normal_nll,
            skewness
        ]

    return out


rng = np.random.default_rng(
    RANDOM_STATE
)

Xtr = []
Xte = []
ytr = []
yte = []

train_meta_extra = []
test_meta_extra = []

for r in records:

    q = qsignal_extra(
        r['signal']
    )

    train_starts = choose_starts(
        0,
        r['split'],
        TRAIN_WINDOWS_PER_RECORD,
        TEST_WINDOW_SIZE,
        rng
    )

    test_starts = choose_starts(
        r['split'],
        len(q),
        TEST_WINDOWS_PER_RECORD,
        TEST_WINDOW_SIZE,
        rng
    )

    Xtr.append(
        extract_extra_features(
            q,
            train_starts
        )
    )

    Xte.append(
        extract_extra_features(
            q,
            test_starts
        )
    )

    ytr.append(
        np.full(
            len(train_starts),
            r['label'],
            dtype=np.int32
        )
    )

    yte.append(
        np.full(
            len(test_starts),
            r['label'],
            dtype=np.int32
        )
    )

    train_meta_extra += [
        {
            'file': r['path'].name,
            'label': r['label'],
            'rpm': r['rpm'],
            'start': int(s)
        }
        for s in train_starts
    ]

    test_meta_extra += [
        {
            'file': r['path'].name,
            'label': r['label'],
            'rpm': r['rpm'],
            'start': int(s)
        }
        for s in test_starts
    ]


X_train_extra = np.concatenate(
    Xtr
)

X_test_extra = np.concatenate(
    Xte
)

y_train_extra = np.concatenate(
    ytr
)

y_test_extra = np.concatenate(
    yte
)

train_meta_extra = pd.DataFrame(
    train_meta_extra
)

test_meta_extra = pd.DataFrame(
    test_meta_extra
)


# SAME shuffle behavior as main notebook
pt = rng.permutation(
    len(y_train_extra)
)

pv = rng.permutation(
    len(y_test_extra)
)

X_train_extra = X_train_extra[pt]
y_train_extra = y_train_extra[pt]

X_test_extra = X_test_extra[pv]
y_test_extra = y_test_extra[pv]

train_meta_extra = (
    train_meta_extra
    .iloc[pt]
    .reset_index(drop=True)
)

test_meta_extra = (
    test_meta_extra
    .iloc[pv]
    .reset_index(drop=True)
)


# ============================================================
# Feature sets to compare
# ============================================================

ALL_FEATURE_NAMES = [
    'Peak',
    'MAV',
    'ZC',
    'Kurtosis',
    'Normal NLL',
    'Skewness'
]

FEATURE_SETS = {
    'Baseline: Peak + MAV + ZC':
        [0, 1, 2],

    'Baseline + Kurtosis':
        [0, 1, 2, 3],

    'Baseline + Normal NLL':
        [0, 1, 2, 4],

    'Baseline + Skewness':
        [0, 1, 2, 5],

    'Baseline + Kurtosis + Normal NLL':
        [0, 1, 2, 3, 4],

    'Baseline + All 3 New Features':
        [0, 1, 2, 3, 4, 5],
}


results = []

for name, cols in FEATURE_SETS.items():

    Xtr_set = X_train_extra[:, cols]
    Xte_set = X_test_extra[:, cols]

    pipeline = Pipeline([
        (
            'scaler',
            StandardScaler()
        ),
        (
            'mlr',
            LogisticRegression(
                solver='lbfgs',
                max_iter=5000,
                random_state=RANDOM_STATE
            )
        )
    ])

    if PROTOCOL == 'all_loads':

        cv = StratifiedGroupKFold(
            n_splits=4,
            shuffle=True,
            random_state=RANDOM_STATE
        )

        groups = (
            train_meta_extra[
                'file'
            ].to_numpy()
        )

        grid = GridSearchCV(
            pipeline,
            {
                'mlr__C':
                    C_VALUES
            },
            scoring='balanced_accuracy',
            cv=cv,
            n_jobs=-1,
            refit=True
        )

        grid.fit(
            Xtr_set,
            y_train_extra,
            groups=groups
        )

    else:

        cv = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=RANDOM_STATE
        )

        grid = GridSearchCV(
            pipeline,
            {
                'mlr__C':
                    C_VALUES
            },
            scoring='balanced_accuracy',
            cv=cv,
            n_jobs=-1,
            refit=True
        )

        grid.fit(
            Xtr_set,
            y_train_extra
        )

    pred = grid.predict(
        Xte_set
    )

    results.append({
        'Feature Set': name,
        'Num Features': len(cols),
        'Best C': grid.best_params_['mlr__C'],
        'CV Balanced Accuracy':
            grid.best_score_,
        'Test Accuracy':
            accuracy_score(
                y_test_extra,
                pred
            ),
        'Test Balanced Accuracy':
            balanced_accuracy_score(
                y_test_extra,
                pred
            ),
        'Macro F1':
            f1_score(
                y_test_extra,
                pred,
                average='macro'
            )
    })


results_df = pd.DataFrame(
    results
)

baseline_accuracy = (
    results_df.loc[
        results_df[
            'Feature Set'
        ]
        ==
        'Baseline: Peak + MAV + ZC',
        'Test Accuracy'
    ].iloc[0]
)

results_df[
    'Accuracy Change (pp)'
] = (
    (
        results_df[
            'Test Accuracy'
        ]
        -
        baseline_accuracy
    )
    * 100.0
)

print(
    f'Feature settings: '
    f'{TEST_WINDOW_SIZE}-sample / '
    f'{TEST_RAW_BITS}-bit Q4.'
    f'{TEST_RAW_BITS-SAMPLE_INT_BITS}'
)

print(
    f'Train windows: {len(y_train_extra):,}'
)

print(
    f'Test windows:  {len(y_test_extra):,}'
)

display(
    results_df.sort_values(
        'Test Accuracy',
        ascending=False
    ).reset_index(
        drop=True
    )
)

Feature settings: 2048-sample / 16-bit Q4.12
Train windows: 14,000
Test windows:  6,000


,Feature Set,Num Features,Best C,CV Balanced Accuracy,Test Accuracy,Test Balanced Accuracy,Macro F1,Accuracy Change (pp)
0,Baseline + All 3 New Features,6,1000.0,0.905500,0.981000,0.981000,0.980946,1.283333
1,Baseline + Kurtosis + Normal NLL,5,1000.0,0.898857,0.979500,0.979500,0.979402,1.133333
2,Baseline + Normal NLL,4,250.0,0.900000,0.977333,0.977333,0.977173,0.916667
3,Baseline + Kurtosis,4,750.0,0.889571,0.975167,0.975167,0.975008,0.700000
4,Baseline + Skewness,4,300.0,0.897429,0.971500,0.971500,0.971316,0.333333
5,Baseline: Peak + MAV + ZC,3,500.0,0.886786,0.968167,0.968167,0.967972,0.000000
